# IBKR Grid Backtest
conda activate qts

## 1. Fetch Historical Data from IBKR

In [ ]:
import threading
import pandas as pd
import numpy as np
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract

class IBDataFetcher(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self.data = []
        self.done = threading.Event()

    def historicalData(self, reqId, bar):
        self.data.append({
            'date':   bar.date,
            'open':   bar.open,
            'high':   bar.high,
            'low':    bar.low,
            'close':  bar.close,
            'volume': bar.volume,
        })

    def historicalDataEnd(self, reqId, start, end):
        self.done.set()

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson=''):
        if errorCode not in (2104, 2106, 2158):   # 忽略连接状态提示
            print(f'[ERROR] reqId={reqId} code={errorCode}: {errorString}')


def get_ibkr_data(symbol, end_date='', duration='2 Y', bar_size='1 day',
                  port=7497, client_id=1):
    """
    从 IBKR 拉取美股历史日线数据，返回 DataFrame (date/open/high/low/close/volume)。
    TWS 或 IB Gateway 必须在后台运行。
    port: 7497=模拟账户, 7496=真实账户
    duration: '1 Y', '2 Y', '6 M' 等
    """
    app = IBDataFetcher()
    app.connect('127.0.0.1', port, clientId=client_id)

    contract = Contract()
    contract.symbol   = symbol
    contract.secType  = 'STK'
    contract.exchange = 'SMART'
    contract.currency = 'USD'

    threading.Thread(target=app.run, daemon=True).start()

    app.reqHistoricalData(
        reqId          = 1,
        contract       = contract,
        endDateTime    = end_date,
        durationStr    = duration,
        barSizeSetting = bar_size,
        whatToShow     = 'TRADES',
        useRTH         = 1,
        formatDate     = 1,
        keepUpToDate   = False,
        chartOptions   = []
    )

    app.done.wait(timeout=30)
    app.disconnect()

    df = pd.DataFrame(app.data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    for col in ['open', 'high', 'low', 'close']:
        df[col] = df[col].astype(float)
    return df

## 2. Load Data & Compute Volatility Params

In [ ]:
from util_Index import volatility_indicators

symbol    = 'AAPL'
startDate = '2023-01-01'
endDate   = '2026-01-01'

# 拉取数据（TWS 需在后台运行）
df = get_ibkr_data(symbol, end_date='', duration='3 Y', bar_size='1 day')
df = df[(df['date'] >= startDate) & (df['date'] <= endDate)].reset_index(drop=True)
print(f'{symbol}: {len(df)} bars  {df["date"].iloc[0].date()} → {df["date"].iloc[-1].date()}')
df.head()

In [ ]:
open_, high, low, close = df['open'], df['high'], df['low'], df['close']
yz, atr_abs, atr_pct, bb, prr, ppercent = volatility_indicators(open_, high, low, close, draw=True)

# Grid 参数
_coverage  = prr.mean() * 0.7
_fee       = 0.0005
min_step   = 2 * _fee
_grid_step = max(atr_pct.iloc[-1] * 1.2, min_step * 2)
_max_layers = int(_coverage / _grid_step)

print(f'coverage={_coverage:.4f}  grid_step={_grid_step:.4f}  max_layers={_max_layers}')

## 3. Backtest

In [ ]:
import numpy as np
import pandas as pd
from util_Grid import Strategy_GridTrading
from util_analyze import trades_to_df, plot_ohlc_box_with_volume, plot_equity_cash_position

class Backtester:
    def __init__(self, df, strategy, initial_cash=10000.0, initial_position=0.0,
                 fee=0.0005, startD=None, endD=None, target_size_mode='all_in'):
        self.df_raw = df.copy()
        self.strategy = strategy
        df_ind = self.strategy.prepare_indicators(self.df_raw)
        self.initial_cash     = float(initial_cash)
        self.initial_position = float(initial_position)
        self.fee  = float(fee)
        self.target_size_mode = target_size_mode

        df_ind['date'] = pd.to_datetime(df_ind['date'])
        if startD: df_ind = df_ind[df_ind['date'] >= pd.Timestamp(startD)]
        if endD:   df_ind = df_ind[df_ind['date'] <= pd.Timestamp(endD)]
        self.df = df_ind

    def _init_state(self):
        return {'cash': self.initial_cash, 'position': self.initial_position,
                'avg_price': np.nan, 'entry_price': np.nan,
                'trades': 0, 'fees_paid': 0.0, 'target_size': 0.0}

    def run(self):
        state = self._init_state()
        trade_log = []
        equity = np.zeros(len(self.df))
        cash   = np.zeros(len(self.df))
        pos    = np.zeros(len(self.df))

        for i, (idx, row) in enumerate(self.df.iterrows()):
            close = float(row['close'])
            if self.target_size_mode == 'all_in':
                state['target_size'] = state['cash'] / close if close > 0 else 0.0
            else:
                state['target_size'] = 1.0
            self.strategy.on_bar(i, row['date'], row, state, fee=self.fee, trade_log=trade_log)
            equity[i] = state['cash'] + state['position'] * close
            cash[i]   = state['cash']
            pos[i]    = state['position']

        out = pd.DataFrame({'equity': equity, 'cash': cash, 'position': pos,
                            'close': self.df['close'].values}, index=self.df.index)
        out.attrs['final_state'] = state
        return out, trade_log

    @staticmethod
    def stats(equity, periods_per_year=252):
        eq  = equity.dropna()
        ret = eq.pct_change().fillna(0.0)
        dd  = eq / eq.cummax() - 1.0
        total = float(eq.iloc[-1] / eq.iloc[0])
        years = (len(eq) - 1) / periods_per_year
        cagr  = total ** (1 / years) - 1 if years > 0 else np.nan
        vol   = ret.std(ddof=0)
        sharpe = (ret.mean() / vol) * np.sqrt(periods_per_year) if vol > 0 else np.nan
        maxdd  = float(dd.min())
        calmar = (cagr / abs(maxdd)) if maxdd < 0 else np.nan
        return {'CAGR': float(cagr), 'Sharpe': float(sharpe),
                'MaxDD': float(maxdd), 'Calmar': float(calmar)}

In [ ]:
strat = Strategy_GridTrading(
    center_n    = 20,
    grid_step   = _grid_step,
    max_layers  = _max_layers,
    stop_pct    = 0.15,
    maxPositionDays = 30
)

bt  = Backtester(df, strat, initial_cash=50000, fee=_fee,
                 startD=startDate, endD=endDate, target_size_mode='all_in')
res, trades = bt.run()

s     = bt.stats(res['equity'])
final = res.attrs['final_state']
print(s)

plot_ohlc_box_with_volume(df, symbol, start=startDate, end=endDate,
                          time_period=1, trade_log=trades)
plot_equity_cash_position(res, s, final, df['date'],
                          start=startDate, end=endDate, strategyName='GridTrading')

## 4. Trade Log

In [ ]:
trades_df = trades_to_df(trades,
    csv_file=f'trade_log_[{symbol}]_{startDate}_{endDate}.csv', savelog=True)
trades_df